# W2-A1 — Vectors & Cosine Similarity (the heart of RAG)

**Watch first (intuition, ~40 min):** 3Blue1Brown — *Essence of Linear Algebra*, chapters 1–4 (vectors, linear combinations, matrix multiplication, dot products). Don't grind the math — just absorb what these *mean*.

**Why this matters:** an **embedding** is just a vector of numbers that represents a piece of text. To find which document is most relevant to a query (the 'R' in RAG = Retrieval), you measure how *similar* two vectors are — and the standard tool is **cosine similarity**. You're about to build it from scratch, no libraries. After this, RAG retrieval is never a black box.

Rules: vectorized NumPy, no `for` loops. No `sklearn` — compute it yourself.

In [10]:
import numpy as np

# --- Given: pretend these are tiny 'embeddings' (don't change them) ---
# Imagine each vector is a document's embedding produced by a model.
query = np.array([1.0, 0.0, 1.0, 0.0])          # the user's question, embedded

docs = np.array([
    [1.0, 0.0, 0.9, 0.1],   # doc 0  (looks a lot like the query)
    [0.0, 1.0, 0.0, 1.0],   # doc 1  (looks nothing like the query)
    [0.8, 0.1, 1.0, 0.0],   # doc 2  (also similar to the query)
    [0.2, 0.9, 0.1, 0.8],   # doc 3  (mostly different)
])

print('query shape:', query.shape)
print('docs shape:', docs.shape)

query shape: (4,)
docs shape: (4, 4)


## Task 1 — Magnitude (length) of a vector
Compute the **magnitude** (a.k.a. L2 norm / length) of the `query` vector — by hand from the definition: square the elements, sum them, square-root.
<br>Then verify your answer matches `np.linalg.norm(query)`.
<br>Hint: `np.sqrt`, `np.sum`, and element-wise `**2`. Finish with an `assert np.allclose(...)`.

In [11]:
# TODO: magnitude of `query` by hand, then assert it equals np.linalg.norm(query)

# By hand from the definition
magnitude = np.sqrt(np.sum(query ** 2))
print('by hand:', magnitude)

# Verify against numpy
print('numpy:  ', np.linalg.norm(query))

assert np.allclose(magnitude, np.linalg.norm(query))

by hand: 1.4142135623730951
numpy:   1.4142135623730951


## Task 2 — Cosine similarity between two vectors
Write a function `cosine_similarity(a, b)` that returns the cosine similarity of two vectors, from the formula:

$$\cos(\theta) = \frac{a \cdot b}{\lVert a \rVert \; \lVert b \rVert}$$

(dot product divided by the product of the two magnitudes). Test it on `query` vs `docs[0]` and `query` vs `docs[1]`.
<br>Hint: you already know the dot product (A1!) and the magnitude (Task 1). The result is between -1 and 1; closer to 1 = more similar.

In [12]:
# TODO: def cosine_similarity(a, b): ...  then test on query vs docs[0] and docs[1]

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print('query vs doc 0:', cosine_similarity(query, docs[0]))  # ~0.969
print('query vs doc 1:', cosine_similarity(query, docs[1]))  # ~0.0


query vs doc 0: 0.9958705948858222
query vs doc 1: 0.0


## Task 3 — Retrieve the most similar document (this IS RAG retrieval)
Compute the cosine similarity of `query` against **every** doc in `docs`, then print which doc index is the best match.
<br>Bonus: rank all docs from most to least similar.
<br>Hint: no loop needed — think about how to apply the formula across all rows at once (dot products: `docs @ query`; norms: `np.linalg.norm(docs, axis=1)`). Then `np.argmax` / `np.argsort` to rank.

In [13]:
# TODO: cosine similarity of query vs ALL docs (vectorized), then find/rank the best match

# Vectorized: all dot products and all norms at once
dots = docs @ query                          # shape (4,)
doc_norms = np.linalg.norm(docs, axis=1)     # shape (4,)
query_norm = np.linalg.norm(query)

similarities = dots / (doc_norms * query_norm)
print('similarities:', similarities)

best = np.argmax(similarities)
print('best match: doc', best)

# Bonus: rank most → least similar
ranking = np.argsort(similarities)[::-1]
print('ranking:', ranking)
for rank, idx in enumerate(ranking):
    print(f'{rank+1}. doc {idx}  (sim={similarities[idx]:.3f})')

similarities: [0.99587059 0.         0.99086739 0.17320508]
best match: doc 0
ranking: [0 2 3 1]
1. doc 0  (sim=0.996)
2. doc 2  (sim=0.991)
3. doc 3  (sim=0.173)
4. doc 1  (sim=0.000)


## Task 4 — Explain (in your own words)
1. What does cosine similarity actually *measure* about two vectors? (Think: angle vs length.)
2. Why do embedding/RAG systems usually prefer **cosine similarity** over plain Euclidean distance? (Hint: what happens to long vs short vectors?)

> _your answer here_
1. What cosine similarity measures. It measures the angle between two vectors, not how long they are. Two vectors pointing in the same direction have cosine 1 regardless of their lengths; perpendicular vectors give 0; opposite directions give -1. Because the formula divides by both magnitudes, the lengths cancel out and only orientation remains. So it's answering "do these point the same way?" rather than "how far apart are their tips?"
2. Why RAG prefers cosine over Euclidean. Embedding magnitudes often carry information you don't want to compare on — a longer document, or one a model happened to encode with larger values, gets a bigger-magnitude vector even if its meaning matches the query. Euclidean distance would penalize that vector just for being long, ranking a semantically perfect but "large" document as far away. Cosine ignores magnitude and compares only direction, which is where embedding models encode meaning. So two texts about the same topic score as similar whether one is a sentence or a page. (In practice many systems also L2-normalize embeddings first, after which cosine similarity and Euclidean distance rank results identically — but cosine is the more robust default when vectors aren't normalized.)